# Rust Intermediate Course
### Deepening Your Rust Knowledge

This course builds on the beginner fundamentals. You should already understand ownership, borrowing, structs, enums, and basic error handling.

**Kernel:** This notebook uses the `evcxr_jupyter` Rust kernel.

---

## Lecture 1: Generics

Generics let you write code that works with multiple types without duplication. Instead of writing separate functions for `i32`, `f64`, and `String`, you write one generic version. The compiler generates specialized code for each concrete type you use — this is called **monomorphization**, and it means generics have zero runtime cost.

You can use generics in functions, structs, enums, and methods. Generic type parameters are conventionally named with single uppercase letters: `T` for type, `E` for error, `K`/`V` for key/value.

Often you need to constrain what a generic type can do — that's where trait bounds come in (covered in Lecture 2). For now, we'll see the basic syntax and how generics eliminate code duplication.

In [ ]:
// Without generics: duplicate code
fn largest_i32(list: &[i32]) -> &i32 {
    let mut largest = &list[0];
    for item in list {
        if item > largest {
            largest = item;
        }
    }
    largest
}

fn largest_f64(list: &[f64]) -> &f64 {
    let mut largest = &list[0];
    for item in list {
        if item > largest {
            largest = item;
        }
    }
    largest
}

println!("Largest i32: {}", largest_i32(&[34, 50, 25, 100, 65]));
println!("Largest f64: {}", largest_f64(&[1.5, 3.7, 2.1, 9.8]));

In [ ]:
// With generics: one function for all comparable types
fn largest<T: PartialOrd>(list: &[T]) -> &T {
    let mut largest = &list[0];
    for item in list {
        if item > largest {
            largest = item;
        }
    }
    largest
}

println!("Largest i32: {}", largest(&[34, 50, 25, 100, 65]));
println!("Largest f64: {}", largest(&[1.5, 3.7, 2.1, 9.8]));
println!("Largest char: {}", largest(&['y', 'm', 'a', 'q']));

In [ ]:
// Generic structs
#[derive(Debug)]
struct Point<T> {
    x: T,
    y: T,
}

// Methods on generic structs
impl<T> Point<T> {
    fn new(x: T, y: T) -> Self {
        Point { x, y }
    }

    fn x(&self) -> &T {
        &self.x
    }
}

// Method only for f64 points
impl Point<f64> {
    fn distance_from_origin(&self) -> f64 {
        (self.x.powi(2) + self.y.powi(2)).sqrt()
    }
}

let int_point = Point::new(5, 10);
let float_point = Point::new(3.0, 4.0);

println!("{:?}, x = {}", int_point, int_point.x());
println!("{:?}, distance = {}", float_point, float_point.distance_from_origin());

In [ ]:
// Multiple generic type parameters
#[derive(Debug)]
struct Pair<T, U> {
    first: T,
    second: U,
}

impl<T, U> Pair<T, U> {
    fn new(first: T, second: U) -> Self {
        Pair { first, second }
    }

    // Mix generic types from different instances
    fn mix<V, W>(self, other: Pair<V, W>) -> Pair<T, W> {
        Pair {
            first: self.first,
            second: other.second,
        }
    }
}

let p1 = Pair::new("hello", 42);
let p2 = Pair::new(3.14, true);
let p3 = p1.mix(p2);
println!("Mixed: {:?}", p3);  // Pair { first: "hello", second: true }

In [ ]:
// Generic enums (you already know these!)
// Option<T> and Result<T, E> are generic enums

// Custom generic enum
#[derive(Debug)]
enum Either<L, R> {
    Left(L),
    Right(R),
}

let a: Either<i32, String> = Either::Left(42);
let b: Either<i32, String> = Either::Right(String::from("hello"));

println!("{:?}", a);
println!("{:?}", b);

### Practice Exercise 1
Create a generic `Stack<T>` struct backed by a `Vec<T>`. Implement `push`, `pop` (returns `Option<T>`), `peek` (returns `Option<&T>`), `is_empty`, and `size`. Test with both `i32` and `String` stacks.

In [ ]:
// Your solution here


---
## Lecture 2: Traits

Traits define shared behavior — they're similar to interfaces in other languages, but more powerful. A trait declares a set of methods that a type must implement. Traits enable polymorphism: you can write code that works with any type that implements a given trait.

Traits can have default method implementations, which types can override. You can use trait bounds to constrain generics, saying "this function works with any type T, as long as T implements Display." The `where` clause provides a cleaner syntax for complex bounds.

Rust's standard library defines many important traits: `Display`, `Debug`, `Clone`, `Copy`, `PartialEq`, `PartialOrd`, `Iterator`, `From`/`Into`, and more. Understanding traits is essential to idiomatic Rust.

In [ ]:
// Defining and implementing a trait
trait Summary {
    fn summarize(&self) -> String;

    // Default implementation
    fn preview(&self) -> String {
        format!("{}...", &self.summarize()[..20.min(self.summarize().len())])
    }
}

#[derive(Debug)]
struct Article {
    title: String,
    author: String,
    content: String,
}

impl Summary for Article {
    fn summarize(&self) -> String {
        format!("{} by {}", self.title, self.author)
    }
}

#[derive(Debug)]
struct Tweet {
    username: String,
    content: String,
}

impl Summary for Tweet {
    fn summarize(&self) -> String {
        format!("@{}: {}", self.username, self.content)
    }
}

let article = Article {
    title: String::from("Rust is Amazing"),
    author: String::from("Alice"),
    content: String::from("Long content here..."),
};

let tweet = Tweet {
    username: String::from("bob"),
    content: String::from("Loving Rust!"),
};

println!("{}", article.summarize());
println!("{}", tweet.summarize());

In [ ]:
// Trait bounds: constrain generic types
fn notify(item: &impl Summary) {
    println!("Breaking news! {}", item.summarize());
}

// Equivalent long form with trait bound syntax
fn notify_verbose<T: Summary>(item: &T) {
    println!("Breaking news! {}", item.summarize());
}

// Multiple trait bounds
fn notify_and_print<T: Summary + std::fmt::Debug>(item: &T) {
    println!("Debug: {:?}", item);
    println!("Summary: {}", item.summarize());
}

let tweet = Tweet {
    username: String::from("alice"),
    content: String::from("Hello world"),
};

notify(&tweet);
notify_and_print(&tweet);

In [ ]:
// where clause for cleaner bounds
fn complex_function<T, U>(t: &T, u: &U) -> String
where
    T: Summary + Clone,
    U: std::fmt::Debug + std::fmt::Display,
{
    format!("T: {} | U: {}", t.summarize(), u)
}

// Returning types that implement traits
fn create_summarizable() -> impl Summary {
    Tweet {
        username: String::from("system"),
        content: String::from("Auto-generated tweet"),
    }
}

let item = create_summarizable();
println!("{}", item.summarize());

In [ ]:
// Important standard library traits
use std::fmt;

#[derive(Debug, Clone, PartialEq)]
struct Color {
    r: u8,
    g: u8,
    b: u8,
}

// Implement Display for custom formatting
impl fmt::Display for Color {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        write!(f, "#{:02X}{:02X}{:02X}", self.r, self.g, self.b)
    }
}

// Implement From for type conversion
impl From<(u8, u8, u8)> for Color {
    fn from(tuple: (u8, u8, u8)) -> Self {
        Color { r: tuple.0, g: tuple.1, b: tuple.2 }
    }
}

let red = Color { r: 255, g: 0, b: 0 };
let also_red = red.clone();
println!("Display: {}", red);
println!("Debug: {:?}", red);
println!("Equal? {}", red == also_red);

let blue: Color = (0, 0, 255).into();  // uses From impl
println!("Blue: {}", blue);

In [ ]:
// Supertraits: requiring one trait to implement another
trait Printable: fmt::Display + fmt::Debug {
    fn print(&self) {
        println!("Display: {}", self);
        println!("Debug: {:?}", self);
    }
}

// Color already implements Display and Debug, so we can:
impl Printable for Color {}

let green = Color { r: 0, g: 255, b: 0 };
green.print();

### Practice Exercise 2
Define a trait `Shape` with methods `area(&self) -> f64` and `perimeter(&self) -> f64`. Implement it for `Circle` (radius) and `Rectangle` (width, height). Write a function `print_shape_info(shape: &impl Shape + std::fmt::Debug)` that prints both. Derive `Debug` for your structs.

In [ ]:
// Your solution here


---
## Lecture 3: Lifetimes

Lifetimes are Rust's way of ensuring that references are always valid. Every reference has a lifetime — the scope for which that reference is valid. Most of the time, the compiler infers lifetimes automatically (lifetime elision), but sometimes you need to annotate them explicitly.

Lifetime annotations don't change how long values live — they describe relationships between lifetimes of multiple references, so the compiler can verify safety. The syntax uses a tick: `'a`, `'b`, etc.

The most common case requiring explicit lifetimes is when a function returns a reference that could come from one of multiple input references. The compiler needs to know which input's lifetime the output is tied to. The special lifetime `'static` means the reference lives for the entire program.

In [ ]:
// Why lifetimes are needed
// This won't compile without lifetime annotations:
// fn longest(x: &str, y: &str) -> &str { ... }
// Which input does the output reference come from?

// With lifetime annotations:
fn longest<'a>(x: &'a str, y: &'a str) -> &'a str {
    if x.len() >= y.len() {
        x
    } else {
        y
    }
}

let string1 = String::from("long string");
let result;
{
    let string2 = String::from("xyz");
    result = longest(string1.as_str(), string2.as_str());
    println!("Longest: {}", result);  // OK: both strings alive here
}

In [ ]:
// Lifetime elision rules (when you DON'T need annotations)
// Rule 1: each reference parameter gets its own lifetime
// Rule 2: if there's one input lifetime, it's assigned to all outputs
// Rule 3: if there's &self or &mut self, self's lifetime is used for outputs

// These functions don't need explicit lifetimes:
fn first_char(s: &str) -> &str {
    &s[..1]  // Rule 2: one input lifetime -> output gets it
}

println!("{}", first_char("hello"));

// This would need annotations (two input references, compiler can't decide):
fn pick_first<'a>(x: &'a str, _y: &str) -> &'a str {
    x
}

println!("{}", pick_first("hello", "world"));

In [ ]:
// Lifetimes in structs
#[derive(Debug)]
struct Excerpt<'a> {
    text: &'a str,
}

impl<'a> Excerpt<'a> {
    fn level(&self) -> i32 {
        3  // no reference returned, no lifetime needed (Rule 3)
    }

    fn announce(&self, announcement: &str) -> &str {
        println!("Attention: {}", announcement);
        self.text  // Rule 3: returns &self lifetime
    }
}

let novel = String::from("Call me Ishmael. Some years ago...");
let first_sentence = novel.split('.').next().unwrap();
let excerpt = Excerpt { text: first_sentence };
println!("Excerpt: {:?}", excerpt);
println!("Level: {}", excerpt.level());
println!("Text: {}", excerpt.announce("New excerpt!"));

In [ ]:
// 'static lifetime: lives for the entire program
let s: &'static str = "I am a string literal";  // stored in binary
println!("{}", s);

// Combining lifetimes, generics, and trait bounds
fn longest_with_announcement<'a, T>(
    x: &'a str,
    y: &'a str,
    ann: T,
) -> &'a str
where
    T: std::fmt::Display,
{
    println!("Announcement: {}", ann);
    if x.len() >= y.len() { x } else { y }
}

let result = longest_with_announcement("hello", "world!", "Comparing strings");
println!("Longest: {}", result);

### Practice Exercise 3
Create a struct `TextProcessor<'a>` that holds a `&'a str`. Implement methods: `word_count(&self) -> usize`, `find_word(&self, word: &str) -> Option<usize>` (returns index), and `substring(&self, start: usize, end: usize) -> &'a str`. Pay attention to which methods need lifetime annotations and which don't.

In [ ]:
// Your solution here


---
## Lecture 4: Closures

Closures are anonymous functions that can capture values from their enclosing scope. They're used extensively with iterators, callbacks, and anywhere you need a short, inline function. Unlike regular functions, closures can infer their parameter and return types from context.

Closures implement one or more of three traits depending on how they capture variables: `Fn` (borrows immutably), `FnMut` (borrows mutably), or `FnOnce` (takes ownership). The compiler determines which trait a closure implements based on how it uses captured values.

The `move` keyword forces a closure to take ownership of all captured variables, which is essential for threading and returning closures from functions.

In [ ]:
// Closure syntax
let add = |a: i32, b: i32| -> i32 { a + b };
let add_short = |a, b| a + b;  // types inferred from usage

println!("add(3, 4) = {}", add(3, 4));
println!("add_short(3, 4) = {}", add_short(3, 4i32));

// Single-expression closures don't need braces
let double = |x| x * 2;
let square = |x: i32| x * x;

println!("double(5) = {}", double(5));
println!("square(5) = {}", square(5));

In [ ]:
// Capturing from environment
let name = String::from("Rust");
let greeting = || println!("Hello, {}!", name);  // borrows name (Fn)
greeting();
greeting();  // can call multiple times
println!("name still valid: {}", name);  // name not consumed

// Mutable capture
let mut count = 0;
let mut increment = || {
    count += 1;  // borrows count mutably (FnMut)
    println!("Count: {}", count);
};
increment();
increment();
increment();

In [ ]:
// move closures: take ownership
let name = String::from("Alice");
let greet = move || println!("Hello, {}!", name);  // name is moved
greet();
// println!("{}", name);  // ERROR: name was moved

// Essential for threads
use std::thread;

let data = vec![1, 2, 3];
let handle = thread::spawn(move || {
    println!("In thread: {:?}", data);
});
handle.join().unwrap();

In [ ]:
// Closures as function parameters
fn apply<F: Fn(i32) -> i32>(f: F, x: i32) -> i32 {
    f(x)
}

fn apply_twice<F: Fn(i32) -> i32>(f: F, x: i32) -> i32 {
    f(f(x))
}

println!("apply double to 5: {}", apply(|x| x * 2, 5));
println!("apply_twice double to 5: {}", apply_twice(|x| x * 2, 5));

// Returning closures (must use Box or impl)
fn make_adder(n: i32) -> impl Fn(i32) -> i32 {
    move |x| x + n
}

let add5 = make_adder(5);
let add10 = make_adder(10);
println!("add5(3) = {}", add5(3));
println!("add10(3) = {}", add10(3));

In [ ]:
// Fn vs FnMut vs FnOnce
fn call_fn<F: Fn()>(f: F) { f(); f(); }  // can call multiple times
fn call_fn_mut<F: FnMut()>(mut f: F) { f(); f(); }  // can call multiple times
fn call_fn_once<F: FnOnce()>(f: F) { f(); }  // can only call once

let s = String::from("hello");
call_fn(|| println!("Fn: {}", s));  // borrows s

let mut counter = 0;
call_fn_mut(|| { counter += 1; println!("FnMut: {}", counter); });

let name = String::from("world");
call_fn_once(move || println!("FnOnce: {}", name));  // consumes name
// name is gone

### Practice Exercise 4
Write a function `filter_map_reduce` that takes a `Vec<i32>`, a filter closure, a map closure, and a reduce closure (with initial value), and applies them in sequence. Test: filter even numbers, double them, sum the result.

In [ ]:
// Your solution here


---
## Lecture 5: Iterators

Iterators are one of Rust's most powerful abstractions. The `Iterator` trait requires implementing just one method: `next()`, which returns `Option<Self::Item>` — `Some(value)` for each element and `None` when exhausted.

Iterator adaptors like `map`, `filter`, `flat_map`, `take`, `skip`, `zip`, `enumerate`, and `chain` transform iterators lazily — no work is done until you consume the iterator with a method like `collect()`, `for_each()`, `fold()`, `sum()`, or `count()`. This lazy evaluation means you can chain many operations without creating intermediate collections.

Iterators in Rust are zero-cost abstractions — the compiler optimizes them to be as fast as hand-written loops, and often faster because they enable additional optimizations like vectorization.

In [ ]:
// The Iterator trait basics
let v = vec![1, 2, 3, 4, 5];

// .iter() borrows, .into_iter() consumes, .iter_mut() borrows mutably
let mut iter = v.iter();
println!("{:?}", iter.next());  // Some(1)
println!("{:?}", iter.next());  // Some(2)
println!("{:?}", iter.next());  // Some(3)

// for loop uses into_iter implicitly
for val in &v {
    print!("{} ", val);
}
println!();

In [ ]:
// Iterator adaptors (lazy — must be consumed)
let v = vec![1, 2, 3, 4, 5, 6, 7, 8, 9, 10];

// map: transform each element
let doubled: Vec<i32> = v.iter().map(|x| x * 2).collect();
println!("Doubled: {:?}", doubled);

// filter: keep elements matching predicate
let evens: Vec<&i32> = v.iter().filter(|x| *x % 2 == 0).collect();
println!("Evens: {:?}", evens);

// chain: combine filter + map
let result: Vec<i32> = v.iter()
    .filter(|x| *x % 2 == 0)
    .map(|x| x * x)
    .collect();
println!("Even squares: {:?}", result);

// filter_map: filter and map in one step
let parsed: Vec<i32> = vec!["1", "two", "3", "four", "5"]
    .iter()
    .filter_map(|s| s.parse::<i32>().ok())
    .collect();
println!("Parsed: {:?}", parsed);

In [ ]:
// Consuming adaptors
let v = vec![1, 2, 3, 4, 5];

let sum: i32 = v.iter().sum();
let product: i32 = v.iter().product();
let count = v.iter().count();
let min = v.iter().min();
let max = v.iter().max();

println!("Sum: {}, Product: {}, Count: {}", sum, product, count);
println!("Min: {:?}, Max: {:?}", min, max);

// fold: reduce to a single value with accumulator
let sum = v.iter().fold(0, |acc, x| acc + x);
let concat = v.iter().fold(String::new(), |acc, x| format!("{}{}", acc, x));
println!("Fold sum: {}", sum);
println!("Fold concat: {}", concat);

// any, all
let has_even = v.iter().any(|x| x % 2 == 0);
let all_positive = v.iter().all(|x| *x > 0);
println!("Has even: {}, All positive: {}", has_even, all_positive);

In [ ]:
// More useful adaptors
let v = vec![1, 2, 3, 4, 5];

// enumerate: get index + value
for (i, val) in v.iter().enumerate() {
    print!("[{}]={} ", i, val);
}
println!();

// zip: combine two iterators
let names = vec!["Alice", "Bob", "Charlie"];
let ages = vec![30, 25, 35];
let people: Vec<_> = names.iter().zip(ages.iter()).collect();
println!("People: {:?}", people);

// take, skip
let first_three: Vec<&i32> = v.iter().take(3).collect();
let skip_two: Vec<&i32> = v.iter().skip(2).collect();
println!("First 3: {:?}", first_three);
println!("Skip 2: {:?}", skip_two);

// chain: concatenate iterators
let a = vec![1, 2];
let b = vec![3, 4];
let combined: Vec<&i32> = a.iter().chain(b.iter()).collect();
println!("Chained: {:?}", combined);

// flat_map: map + flatten
let words: Vec<&str> = vec!["hello world", "foo bar"]
    .iter()
    .flat_map(|s| s.split_whitespace())
    .collect();
println!("Words: {:?}", words);

In [ ]:
// Custom iterator
struct Counter {
    count: u32,
    max: u32,
}

impl Counter {
    fn new(max: u32) -> Counter {
        Counter { count: 0, max }
    }
}

impl Iterator for Counter {
    type Item = u32;

    fn next(&mut self) -> Option<Self::Item> {
        if self.count < self.max {
            self.count += 1;
            Some(self.count)
        } else {
            None
        }
    }
}

// Now we get all iterator methods for free!
let sum: u32 = Counter::new(5).sum();
println!("Counter sum: {}", sum);

let pairs: Vec<(u32, u32)> = Counter::new(5)
    .zip(Counter::new(5).skip(1))
    .collect();
println!("Counter pairs: {:?}", pairs);

let result: u32 = Counter::new(5)
    .zip(Counter::new(5).skip(1))
    .map(|(a, b)| a * b)
    .filter(|x| x % 2 == 0)
    .sum();
println!("Complex chain: {}", result);

### Practice Exercise 5
Implement a `Fibonacci` struct that implements `Iterator`. It should yield the infinite Fibonacci sequence (0, 1, 1, 2, 3, 5, 8...). Then use iterator adaptors to: (1) take the first 20 Fibonacci numbers, (2) filter only the even ones, (3) sum them.

In [ ]:
// Your solution here


---
## Lecture 6: Smart Pointers

Smart pointers are data structures that act like pointers but have additional metadata and capabilities. They implement the `Deref` trait (to behave like references) and the `Drop` trait (to run cleanup code when they go out of scope).

`Box<T>` allocates data on the heap — useful for large data, recursive types, or trait objects. `Rc<T>` (Reference Counted) enables multiple ownership through reference counting. `RefCell<T>` provides interior mutability — the ability to mutate data even through an immutable reference, with borrowing rules checked at runtime instead of compile time.

These types are essential building blocks. `Rc<RefCell<T>>` is a common pattern for shared mutable state in single-threaded code.

In [ ]:
// Box<T>: heap allocation
let b = Box::new(5);
println!("Boxed value: {}", b);  // Deref makes it transparent

// Useful for recursive types
#[derive(Debug)]
enum List {
    Cons(i32, Box<List>),
    Nil,
}

use List::{Cons, Nil};

let list = Cons(1, Box::new(Cons(2, Box::new(Cons(3, Box::new(Nil))))));
println!("{:?}", list);

// Box for trait objects (dynamic dispatch)
trait Animal {
    fn speak(&self) -> &str;
}

struct Dog;
struct Cat;

impl Animal for Dog {
    fn speak(&self) -> &str { "Woof!" }
}

impl Animal for Cat {
    fn speak(&self) -> &str { "Meow!" }
}

let animals: Vec<Box<dyn Animal>> = vec![Box::new(Dog), Box::new(Cat)];
for animal in &animals {
    println!("{}", animal.speak());
}

In [ ]:
// Deref trait: custom dereference behavior
use std::ops::Deref;

struct MyBox<T>(T);

impl<T> MyBox<T> {
    fn new(x: T) -> MyBox<T> {
        MyBox(x)
    }
}

impl<T> Deref for MyBox<T> {
    type Target = T;

    fn deref(&self) -> &T {
        &self.0
    }
}

let x = 5;
let y = MyBox::new(x);
assert_eq!(5, *y);  // deref coercion

// Deref coercion: &MyBox<String> -> &String -> &str
fn hello(name: &str) {
    println!("Hello, {}!", name);
}

let m = MyBox::new(String::from("Rust"));
hello(&m);  // Deref chain: &MyBox<String> -> &String -> &str

In [ ]:
// Rc<T>: reference counted pointer (single-threaded)
use std::rc::Rc;

#[derive(Debug)]
enum SharedList {
    Cons(i32, Rc<SharedList>),
    Nil,
}

let a = Rc::new(SharedList::Cons(5, Rc::new(SharedList::Cons(10, Rc::new(SharedList::Nil)))));
println!("Reference count after a: {}", Rc::strong_count(&a));

let b = SharedList::Cons(3, Rc::clone(&a));  // increment ref count
println!("Reference count after b: {}", Rc::strong_count(&a));

{
    let c = SharedList::Cons(4, Rc::clone(&a));
    println!("Reference count after c: {}", Rc::strong_count(&a));
}

println!("Reference count after c drops: {}", Rc::strong_count(&a));

In [ ]:
// RefCell<T>: interior mutability (runtime borrow checking)
use std::cell::RefCell;

let data = RefCell::new(vec![1, 2, 3]);

// Borrow immutably
println!("Data: {:?}", data.borrow());

// Borrow mutably (even though `data` is not `mut`!)
data.borrow_mut().push(4);
println!("After push: {:?}", data.borrow());

// Rc<RefCell<T>>: shared mutable state
let shared = Rc::new(RefCell::new(0));

let a = Rc::clone(&shared);
let b = Rc::clone(&shared);

*a.borrow_mut() += 10;
*b.borrow_mut() += 20;

println!("Shared value: {}", shared.borrow());  // 30

### Practice Exercise 6
Create a simple tree structure using `Rc<RefCell<T>>`. Each `TreeNode` has a value (i32), and a `Vec<Rc<RefCell<TreeNode>>>` of children. Implement a method to add children and a method to print the tree (with indentation).

In [ ]:
// Your solution here


---
## Lecture 7: Concurrency

Rust's ownership system prevents data races at compile time, making concurrent programming much safer than in most languages. If your code compiles, you know it's free from data races — the compiler enforces the `Send` and `Sync` traits automatically.

Rust supports two main concurrency paradigms: **message passing** (channels, like Go's goroutines) and **shared state** (mutexes). The `std::sync::mpsc` channel lets threads communicate by sending values. `Mutex<T>` provides mutual exclusion, and `Arc<T>` (Atomic Reference Count) is the thread-safe version of `Rc<T>`.

For most use cases, consider the `rayon` crate for data parallelism or `tokio` for async I/O — but understanding the standard library primitives is essential.

In [ ]:
// Spawning threads
use std::thread;
use std::time::Duration;

let handle = thread::spawn(|| {
    for i in 1..5 {
        println!("  spawned thread: {}", i);
        thread::sleep(Duration::from_millis(1));
    }
});

for i in 1..3 {
    println!("main thread: {}", i);
    thread::sleep(Duration::from_millis(1));
}

handle.join().unwrap();  // wait for thread to finish
println!("Both threads done!");

In [ ]:
// move closures with threads
let data = vec![1, 2, 3];

let handle = thread::spawn(move || {
    println!("Thread got data: {:?}", data);
    data.iter().sum::<i32>()
});

let result = handle.join().unwrap();
println!("Sum from thread: {}", result);

// Multiple threads
let mut handles = vec![];
for i in 0..5 {
    handles.push(thread::spawn(move || {
        println!("Thread {} running", i);
        i * i
    }));
}

let results: Vec<i32> = handles.into_iter()
    .map(|h| h.join().unwrap())
    .collect();
println!("Results: {:?}", results);

In [ ]:
// Message passing with channels
use std::sync::mpsc;

let (tx, rx) = mpsc::channel();

thread::spawn(move || {
    let messages = vec!["hello", "from", "the", "thread"];
    for msg in messages {
        tx.send(msg).unwrap();
        thread::sleep(Duration::from_millis(10));
    }
});

// Receive messages
for received in rx {
    println!("Got: {}", received);
}

In [ ]:
// Multiple producers
let (tx, rx) = mpsc::channel();

for i in 0..3 {
    let tx_clone = tx.clone();
    thread::spawn(move || {
        tx_clone.send(format!("Message from thread {}", i)).unwrap();
    });
}
drop(tx);  // drop original sender so rx knows when all senders are done

for msg in rx {
    println!("{}", msg);
}

In [ ]:
// Shared state: Mutex + Arc
use std::sync::{Arc, Mutex};

let counter = Arc::new(Mutex::new(0));
let mut handles = vec![];

for _ in 0..10 {
    let counter = Arc::clone(&counter);
    let handle = thread::spawn(move || {
        let mut num = counter.lock().unwrap();
        *num += 1;
    });
    handles.push(handle);
}

for handle in handles {
    handle.join().unwrap();
}

println!("Counter: {}", *counter.lock().unwrap());  // 10

### Practice Exercise 7
Create a parallel word counter: split a large text into chunks, spawn a thread for each chunk to count word frequencies (HashMap), then merge the results using a channel. Test with a paragraph of text.

In [ ]:
// Your solution here


---
## Lecture 8: Useful Libraries Part 1

Rust's standard library is intentionally minimal — the ecosystem relies on community crates. Here we cover essential crates you'll use in almost every project. Note: in evcxr you can add crates with `:dep crate_name = "version"`, but some may not work in the notebook environment.

**serde** is the serialization/deserialization framework — it works with JSON, TOML, YAML, and many other formats. **clap** builds command-line argument parsers with derive macros. **regex** provides regular expressions. **log** + **env_logger** handle logging.

These examples show the API patterns. In a real project, you'd add them to `Cargo.toml`.

In [ ]:
// serde + serde_json: Serialization
// In Cargo.toml: serde = { version = "1", features = ["derive"] }
//                serde_json = "1"

// Example code (may need :dep in evcxr):
// :dep serde = { version = "1", features = ["derive"] }
// :dep serde_json = "1"

// use serde::{Serialize, Deserialize};

// #[derive(Serialize, Deserialize, Debug)]
// struct Person {
//     name: String,
//     age: u32,
//     #[serde(default)]
//     email: Option<String>,
// }

// let person = Person {
//     name: "Alice".to_string(),
//     age: 30,
//     email: Some("alice@example.com".to_string()),
// };

// // Serialize to JSON
// let json = serde_json::to_string_pretty(&person).unwrap();
// println!("JSON:\n{}", json);

// // Deserialize from JSON
// let json_str = r#"{"name": "Bob", "age": 25}"#;
// let bob: Person = serde_json::from_str(json_str).unwrap();
// println!("Deserialized: {:?}", bob);

println!("serde example — uncomment above and add :dep to run");
println!("serde is the #1 most downloaded Rust crate");
println!("Key derive macros: #[derive(Serialize, Deserialize)]");
println!("Attributes: #[serde(rename = ...)], #[serde(default)], #[serde(skip)]");

In [ ]:
// clap: Command-line argument parsing
// In Cargo.toml: clap = { version = "4", features = ["derive"] }

// use clap::Parser;

// #[derive(Parser, Debug)]
// #[command(name = "myapp", about = "A sample CLI app")]
// struct Args {
//     /// Name of the person to greet
//     #[arg(short, long)]
//     name: String,
//
//     /// Number of times to greet
//     #[arg(short, long, default_value_t = 1)]
//     count: u8,
//
//     /// Enable verbose output
//     #[arg(short, long)]
//     verbose: bool,
// }

// fn main() {
//     let args = Args::parse();
//     for _ in 0..args.count {
//         println!("Hello, {}!", args.name);
//     }
// }

// Usage: myapp --name Alice --count 3 --verbose

println!("clap example — this is a main() pattern for CLI apps");
println!("Use #[derive(Parser)] for derive-based argument parsing");
println!("Supports subcommands, validation, shell completions");

In [ ]:
// regex: Regular expressions
// In Cargo.toml: regex = "1"

// :dep regex = "1"
// use regex::Regex;

// let re = Regex::new(r"\b(\w+)@(\w+\.\w+)\b").unwrap();
// let text = "Contact alice@example.com or bob@test.org";

// // Find all matches
// for cap in re.captures_iter(text) {
//     println!("Full: {}, User: {}, Domain: {}", &cap[0], &cap[1], &cap[2]);
// }

// // Replace
// let result = re.replace_all(text, "[REDACTED]");
// println!("Redacted: {}", result);

// // is_match
// println!("Has email? {}", re.is_match(text));

println!("regex example — uncomment above and add :dep to run");
println!("Key types: Regex, Captures, Match");
println!("Methods: is_match(), find(), captures(), replace_all()");

In [ ]:
// log + env_logger: Structured logging
// In Cargo.toml: log = "0.4"
//                env_logger = "0.10"

// use log::{info, warn, error, debug, trace};

// fn main() {
//     env_logger::init();  // reads RUST_LOG env var
//
//     error!("This is an error");
//     warn!("This is a warning");
//     info!("This is info");
//     debug!("This is debug");
//     trace!("This is trace");
//
//     info!("Processing {} items", 42);
// }

// Run with: RUST_LOG=debug cargo run
// Filter: RUST_LOG=myapp=debug,other_crate=warn cargo run

println!("log + env_logger example — for CLI/server applications");
println!("Set RUST_LOG=debug to see debug messages");
println!("Levels: error > warn > info > debug > trace");
println!("Consider 'tracing' crate for async/structured logging");

### Practice Exercise 8
Design (on paper or in comments) a CLI tool using clap that processes a CSV file. Define the Args struct with: `input` (required file path), `output` (optional, defaults to stdout), `delimiter` (optional, default comma), `--header` flag. Show the struct definition and how you'd use each argument.

In [ ]:
// Your solution here


---
## Lecture 9: Useful Libraries Part 2

This lecture covers crates for networking, async programming, and common utilities. **reqwest** is the go-to HTTP client. **tokio** is the dominant async runtime. **chrono** handles dates and times. **rand** provides random number generation.

Async Rust is a large topic (covered in depth in the Expert course), but knowing the basics of tokio — `#[tokio::main]`, `.await`, and spawning tasks — is practical knowledge you'll use frequently.

These crates form the backbone of most Rust applications, from CLI tools to web servers.

In [ ]:
// reqwest: HTTP client
// In Cargo.toml: reqwest = { version = "0.11", features = ["json", "blocking"] }

// Blocking (synchronous) example:
// let body = reqwest::blocking::get("https://httpbin.org/get")
//     .unwrap()
//     .text()
//     .unwrap();
// println!("{}", body);

// Async example (with tokio):
// async fn fetch() -> Result<(), reqwest::Error> {
//     let resp = reqwest::get("https://httpbin.org/json")
//         .await?
//         .json::<serde_json::Value>()
//         .await?;
//     println!("{:#?}", resp);
//     Ok(())
// }

// POST with JSON:
// let client = reqwest::Client::new();
// let resp = client.post("https://httpbin.org/post")
//     .json(&serde_json::json!({"key": "value"}))
//     .send()
//     .await?;

println!("reqwest: the most popular HTTP client for Rust");
println!("Supports: async/blocking, JSON, forms, headers, cookies");
println!("Use 'blocking' feature for synchronous code");

In [ ]:
// tokio: async runtime
// In Cargo.toml: tokio = { version = "1", features = ["full"] }

// #[tokio::main]
// async fn main() {
//     // Spawn concurrent tasks
//     let task1 = tokio::spawn(async {
//         tokio::time::sleep(Duration::from_millis(100)).await;
//         println!("Task 1 done");
//         42
//     });
//
//     let task2 = tokio::spawn(async {
//         tokio::time::sleep(Duration::from_millis(50)).await;
//         println!("Task 2 done");
//         "hello"
//     });
//
//     // Both run concurrently
//     let (r1, r2) = tokio::join!(task1, task2);
//     println!("Results: {:?}, {:?}", r1, r2);
// }

println!("tokio: the most popular async runtime for Rust");
println!("Key features: spawn, join!, select!, channels, timers");
println!("Use #[tokio::main] on your main function");
println!("Great for I/O-bound tasks: HTTP, databases, file I/O");

In [ ]:
// rand: random number generation
// :dep rand = "0.8"

// use rand::Rng;
// use rand::seq::SliceRandom;

// let mut rng = rand::thread_rng();

// // Random numbers
// let n: i32 = rng.gen_range(1..=100);
// println!("Random 1-100: {}", n);

// let f: f64 = rng.gen();  // 0.0 to 1.0
// println!("Random float: {}", f);

// let b: bool = rng.gen();
// println!("Random bool: {}", b);

// // Shuffle a vector
// let mut v = vec![1, 2, 3, 4, 5];
// v.shuffle(&mut rng);
// println!("Shuffled: {:?}", v);

// // Choose random element
// let choice = v.choose(&mut rng);
// println!("Random choice: {:?}", choice);

println!("rand: random number generation");
println!("gen_range(1..=100), gen::<f64>(), gen::<bool>()");
println!("SliceRandom: shuffle(), choose(), choose_multiple()");

In [ ]:
// anyhow + thiserror: Error handling in applications vs libraries

// anyhow: for applications (easy, flexible errors)
// use anyhow::{Context, Result};
//
// fn read_config() -> Result<String> {
//     let content = std::fs::read_to_string("config.toml")
//         .context("Failed to read config file")?;
//     Ok(content)
// }

// thiserror: for libraries (structured error types)
// use thiserror::Error;
//
// #[derive(Error, Debug)]
// enum DataError {
//     #[error("IO error: {0}")]
//     Io(#[from] std::io::Error),
//
//     #[error("Parse error at line {line}: {message}")]
//     Parse { line: usize, message: String },
//
//     #[error("Not found: {0}")]
//     NotFound(String),
// }

println!("anyhow: use in applications for easy error handling");
println!("thiserror: use in libraries for structured error types");
println!("anyhow::Result<T> = Result<T, anyhow::Error>");
println!(".context() adds human-readable context to errors");

### Practice Exercise 9
Design (in comments/pseudocode) an async function using tokio and reqwest that fetches 5 URLs concurrently using `tokio::join!` or `futures::future::join_all`, collects the status codes, and returns them as a `Vec<u16>`. Include error handling with `anyhow`.

In [ ]:
// Your solution here


---
## Lecture 10: Testing & Documentation

Rust has first-class support for testing built into the language and toolchain. Unit tests live in the same file as the code they test, inside a `#[cfg(test)]` module. Integration tests go in a `tests/` directory. The `#[test]` attribute marks test functions, and `assert!`, `assert_eq!`, `assert_ne!` are the primary assertion macros.

Documentation in Rust uses `///` doc comments with Markdown formatting. Code examples in doc comments are automatically run as tests (`cargo test` runs them). This means your documentation examples are always verified to compile and produce correct results.

`cargo test` runs all tests (unit, integration, and doc tests) in parallel by default. You can filter tests by name, run ignored tests, and control parallelism.

In [ ]:
// Unit tests: same file as the code
fn add(a: i32, b: i32) -> i32 {
    a + b
}

fn divide(a: f64, b: f64) -> Result<f64, String> {
    if b == 0.0 {
        Err("Division by zero".to_string())
    } else {
        Ok(a / b)
    }
}

// In a real file, these would be inside #[cfg(test)] mod tests { ... }
// Here we just demonstrate the assertions

// assert_eq! checks equality
assert_eq!(add(2, 3), 5);
println!("add(2, 3) == 5: PASS");

// assert_ne! checks inequality
assert_ne!(add(2, 3), 6);
println!("add(2, 3) != 6: PASS");

// assert! checks a boolean
assert!(add(2, 3) > 0);
println!("add(2, 3) > 0: PASS");

// Testing Results
assert!(divide(10.0, 2.0).is_ok());
assert!(divide(10.0, 0.0).is_err());
assert_eq!(divide(10.0, 2.0).unwrap(), 5.0);
println!("divide tests: PASS");

In [ ]:
// Test organization pattern (how it looks in a real .rs file)

// pub fn is_even(n: i32) -> bool {
//     n % 2 == 0
// }
//
// pub fn factorial(n: u32) -> u64 {
//     (1..=n as u64).product()
// }
//
// #[cfg(test)]  // only compiled when running tests
// mod tests {
//     use super::*;  // import from parent module
//
//     #[test]
//     fn test_is_even() {
//         assert!(is_even(2));
//         assert!(is_even(0));
//         assert!(!is_even(3));
//     }
//
//     #[test]
//     fn test_factorial() {
//         assert_eq!(factorial(0), 1);
//         assert_eq!(factorial(1), 1);
//         assert_eq!(factorial(5), 120);
//     }
//
//     #[test]
//     #[should_panic(expected = "overflow")]
//     fn test_overflow() {
//         // test that a panic occurs
//         panic!("overflow");
//     }
//
//     #[test]
//     #[ignore]  // skip unless explicitly requested
//     fn expensive_test() {
//         // slow test
//     }
// }

println!("Test file structure demonstrated in comments above");
println!("Run: cargo test");
println!("Run specific: cargo test test_factorial");
println!("Run ignored: cargo test -- --ignored");
println!("Show output: cargo test -- --nocapture");

In [ ]:
// Doc comments and doc tests

// /// Adds two numbers together.
// ///
// /// # Examples
// ///
// /// ```
// /// let result = mylib::add(2, 3);
// /// assert_eq!(result, 5);
// /// ```
// ///
// /// # Panics
// ///
// /// This function doesn't panic.
// pub fn add(a: i32, b: i32) -> i32 {
//     a + b
// }

// The code in ``` blocks is automatically tested by cargo test!

// Common doc sections:
// # Examples     - usage examples (tested!)
// # Panics       - when/why it panics
// # Errors       - what errors it returns (for Result)
// # Safety       - for unsafe functions

// Generate docs: cargo doc --open

println!("Doc comment sections: Examples, Panics, Errors, Safety");
println!("Code in /// ``` blocks are tested by cargo test");
println!("Generate HTML docs: cargo doc --open");

In [ ]:
// Integration tests (in tests/ directory)

// File: tests/integration_test.rs
//
// use mylib;  // your crate
//
// #[test]
// fn test_full_workflow() {
//     let result = mylib::process("input data");
//     assert!(result.is_ok());
//     let output = result.unwrap();
//     assert_eq!(output.status, "success");
// }

// Shared test helpers: tests/common/mod.rs
// (use mod.rs so cargo doesn't treat it as a test file)

// Project structure with tests:
// my_project/
// ├── src/
// │   ├── lib.rs          # library code
// │   └── main.rs         # binary
// ├── tests/
// │   ├── common/
// │   │   └── mod.rs      # shared test helpers
// │   ├── integration_a.rs
// │   └── integration_b.rs
// └── benches/
//     └── benchmark.rs    # benchmarks

println!("Integration tests go in tests/ directory");
println!("Each file in tests/ is compiled as a separate crate");
println!("Shared helpers go in tests/common/mod.rs");

### Practice Exercise 10
Write a `Calculator` struct with methods `add`, `subtract`, `multiply`, `divide` (returns Result). Write comprehensive unit tests covering: normal cases, edge cases (divide by zero), and property-based assertions (e.g., `add(a, b) == add(b, a)`). Include doc comments with examples for each method.

---

## Congratulations!
You've completed the Intermediate Rust Course! You now understand:
- Generics and monomorphization
- Traits, trait bounds, and standard library traits
- Lifetime annotations and elision rules
- Closures and the Fn traits
- Iterators and iterator adaptors
- Smart pointers (Box, Rc, RefCell)
- Concurrency with threads, channels, and mutexes
- Essential crates (serde, clap, regex, tokio, reqwest)
- Testing and documentation

**Next:** Move on to the **Expert Rust Course** for advanced traits, unsafe Rust, macros, async deep-dive, design patterns, and frameworks!

In [ ]:
// Your solution here
